<a href="https://colab.research.google.com/github/jeneferTodt/analise-preditiva-entregas/blob/feat%2Fpipeline-dados/salesinsight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Introdução


## SalesInsight PY — Análise e Visualização de Dados de Vendas

Este projeto tem como objetivo aplicar técnicas de programação e análise de dados utilizando Python para realizar a inspeção, limpeza, transformação e análise de um dataset de vendas.

O fluxo contempla o cálculo de métricas, a segmentação de clientes, a geração de gráficos e a exportação de resultados em arquivos CSV e JSON, facilitando a interpretação dos dados e a identificação de informações relevantes para o negócio.

**Tecnologias utilizadas:** Python, Pandas, NumPy, Matplotlib e Seaborn.

**Ambiente de desenvolvimento:** Google Colab.

# 02 — Imports

In [165]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from datetime import datetime, timedelta



# 03 — Dataset

In [136]:
## O gerador abaixo cria propositalmente dados “sujos”, que servirão de matéria-prima para o requisito de limpeza. Use-o como está:


import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

def gerar_dataset_vendas(n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]

    data_inicio = datetime(2025, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")
        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # --- sujeira proposital para a etapa de limpeza ---
        if random.random() < 0.03:
        	data_txt = "DATA INVALIDA"       	# data invalida

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco,
        })

    return pd.DataFrame(dados)

# Gerar e salvar o CSV bruto
df_bruto = gerar_dataset_vendas()
df_bruto.to_csv("vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")
print(df_bruto.head())


Dataset gerado com 200 registros.
   id_venda     data_venda      cliente     produto     categoria    regiao  \
0         1     2025-05-21  Cliente_016       Mouse   Perifericos   Sudeste   
1         2  DATA INVALIDA  Cliente_003       Mouse   Perifericos       Sul   
2         3     2025-10-15  Cliente_013  Smartphone     Celulares     Norte   
3         4     2025-05-23  Cliente_001     Monitor  Computadores       Sul   
4         5     2025-03-21  Cliente_014       Mouse   Perifericos  Nordeste   

   quantidade  preco_unitario  
0           2          102.90  
1           9          105.13  
2           9         2267.33  
3           4         1181.72  
4           7          114.25  


In [137]:
#  EXPORTAÇÃO DO DATASET

df_bruto.to_csv(
    "vendas.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dataset salvo como vendas.csv")

# Conferir se o arquivo pode ser lido novamente
df_verificacao = pd.read_csv("vendas.csv")

display(df_verificacao.head())
print("Dimensões do arquivo:", df_verificacao.shape)

Dataset salvo como vendas.csv


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,Cliente_016,Mouse,Perifericos,Sudeste,2,102.90
1,2,DATA INVALIDA,Cliente_003,Mouse,Perifericos,Sul,9,105.13
2,3,2025-10-15,Cliente_013,Smartphone,Celulares,Norte,9,2267.33
3,4,2025-05-23,Cliente_001,Monitor,Computadores,Sul,4,1181.72
4,5,2025-03-21,Cliente_014,Mouse,Perifericos,Nordeste,7,114.25


Dimensões do arquivo: (200, 8)


In [138]:
from google.colab import files

files.download("vendas.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#  04- Inspeção

In [139]:
## Descrever a estrutura do DataFrame

def inspecionar_dados(df_bruto):
    """Exibe as informações estruturais do DataFrame."""

    print("\n=== INSPEÇÃO INICIAL DO DATASET ===")
    print(f"Dimensões: {df_bruto.shape}")
    print(f"\nColunas: {list(df_bruto.columns)}")
    print(f"\nTipos de dados:\n{df_bruto.dtypes}")
    print(f"\nValores ausentes por coluna:\n{df_bruto.isnull().sum()}")
    print(f"\nPrimeiros registros:\n{df_bruto.head()}")

    return df_bruto


In [140]:
inspecionar_dados(df_bruto)


=== INSPEÇÃO INICIAL DO DATASET ===
Dimensões: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade          int64
preco_unitario    float64
dtype: object

Valores ausentes por coluna:
id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        0
preco_unitario    0
dtype: int64

Primeiros registros:
   id_venda     data_venda      cliente     produto     categoria    regiao  \
0         1     2025-05-21  Cliente_016       Mouse   Perifericos   Sudeste   
1         2  DATA INVALIDA  Cliente_003       Mouse   Perifericos       Sul   
2         3     2025-10-15  Cliente_013  Smartphone     Celulares     Norte   
3         4     2025-05-23  Cliente_001     Monit

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,Cliente_016,Mouse,Perifericos,Sudeste,2,102.90
1,2,DATA INVALIDA,Cliente_003,Mouse,Perifericos,Sul,9,105.13
2,3,2025-10-15,Cliente_013,Smartphone,Celulares,Norte,9,2267.33
3,4,2025-05-23,Cliente_001,Monitor,Computadores,Sul,4,1181.72
4,5,2025-03-21,Cliente_014,Mouse,Perifericos,Nordeste,7,114.25
...,...,...,...,...,...,...,...,...
195,196,2025-04-21,Cliente_028,Headset,Perifericos,Sul,9,373.19
196,197,2025-07-24,Cliente_027,Monitor,Computadores,Nordeste,6,1317.87
197,198,2025-07-11,Cliente_010,Monitor,Computadores,Centro-Oeste,6,1259.44
198,199,2025-02-17,Cliente_028,Notebook,Computadores,Nordeste,2,3848.36


# 05 - Limpeza e preparação

In [141]:

def tratar_dados(df):
    """
    Trata o DataFrame de vendas.

    Retorna:
        df_limpo: DataFrame tratado.
        relatorio: dicionário com as contagens da limpeza.
    """

    # Criar uma cópia para preservar o DataFrame original
    df_limpo = df.copy()

    # Registrar a quantidade inicial de registros
    registros_iniciais = len(df_limpo)

    # Contabilizar conversões e alterações
    relatorio = {
        "registros_iniciais": registros_iniciais,
        "textos_padronizados": 0,
        "datas_invalidas_removidas": 0,
        "quantidade_convertida": 0,
        "preco_unitario_convertido": 0,
        "registros_sem_alteracao": 0,
        "registros_finais": 0
    }

    # 1. Remover espaços extras nas colunas de texto
    colunas_texto = df_limpo.select_dtypes(
        include=["object", "string"]
    ).columns

    for coluna in colunas_texto:
        valores_antes = df_limpo[coluna].copy()

        df_limpo[coluna] = df_limpo[coluna].str.strip()

        relatorio["textos_padronizados"] += int(
            (
                valores_antes.notna()
                & valores_antes.ne(df_limpo[coluna])
            ).sum()
        )

    # 2. Converter datas e remover registros inválidos
    datas_antes = df_limpo["data_venda"].copy()

    df_limpo["data_venda"] = pd.to_datetime(
        df_limpo["data_venda"],
        errors="coerce"
    )

    datas_invalidas = df_limpo["data_venda"].isna()

    relatorio["datas_invalidas_removidas"] = int(
        datas_invalidas.sum()
    )

    df_limpo = df_limpo.loc[~datas_invalidas].copy()

    # 3. Converter quantidade para inteiro
    quantidade_antes = df_limpo["quantidade"].copy()

    quantidade_convertida = pd.to_numeric(
        quantidade_antes,
        errors="coerce"
    )

    quantidade_valida = (
        quantidade_convertida.notna()
        & (quantidade_convertida % 1 == 0)
    )

    # Não descartar silenciosamente valores inválidos
    if not quantidade_valida.all():
        raise ValueError(
            "Existem valores inválidos ou não inteiros "
            "na coluna quantidade."
        )

    relatorio["quantidade_convertida"] = int(
        (
            quantidade_antes.astype("string")
            != quantidade_convertida.astype("string")
        ).sum()
    )

    df_limpo["quantidade"] = quantidade_convertida.astype(int)

    # 4. Converter preco_unitario para float
    preco_antes = df_limpo["preco_unitario"].copy()

    preco_convertido = pd.to_numeric(
        preco_antes,
        errors="coerce"
    )

    preco_valido = preco_convertido.notna()

    if not preco_valido.all():
        raise ValueError(
            "Existem valores inválidos na coluna preco_unitario."
        )

    relatorio["preco_unitario_convertido"] = int(
        (
            preco_antes.astype("string")
            != preco_convertido.astype("string")
        ).sum()
    )

    df_limpo["preco_unitario"] = preco_convertido.astype(float)

    # 5. Calcular registros sem alteração
    registros_removidos = registros_iniciais - len(df_limpo)

    registros_com_alteracao = (
        relatorio["textos_padronizados"] > 0
        or relatorio["quantidade_convertida"] > 0
        or relatorio["preco_unitario_convertido"] > 0
        or relatorio["datas_invalidas_removidas"] > 0
    )

    relatorio["registros_finais"] = len(df_limpo)

    # Relatório no console
    print("=== RELATÓRIO DE LIMPEZA ===")
    print(f"Registros iniciais: {registros_iniciais}")
    print(
        "Espaços removidos em valores de texto: "
        f"{relatorio['textos_padronizados']}"
    )
    print(
        "Datas inválidas removidas: "
        f"{relatorio['datas_invalidas_removidas']}"
    )
    print(
        "Valores convertidos em quantidade: "
        f"{relatorio['quantidade_convertida']}"
    )
    print(
        "Valores convertidos em preco_unitario: "
        f"{relatorio['preco_unitario_convertido']}"
    )
    print(f"Registros finais: {relatorio['registros_finais']}")

    return df_limpo, relatorio


In [142]:

df_limpo, relatorio = tratar_dados(df_bruto)

display(df_limpo.head())

print("\nTipos de dados após a limpeza:")
print(df_limpo.dtypes)


=== RELATÓRIO DE LIMPEZA ===
Registros iniciais: 200
Espaços removidos em valores de texto: 0
Datas inválidas removidas: 6
Valores convertidos em quantidade: 0
Valores convertidos em preco_unitario: 0
Registros finais: 194


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,Cliente_016,Mouse,Perifericos,Sudeste,2,102.90
2,3,2025-10-15,Cliente_013,Smartphone,Celulares,Norte,9,2267.33
3,4,2025-05-23,Cliente_001,Monitor,Computadores,Sul,4,1181.72
4,5,2025-03-21,Cliente_014,Mouse,Perifericos,Nordeste,7,114.25
5,6,2025-07-03,Cliente_023,Notebook,Computadores,Sudeste,2,3373.92



Tipos de dados após a limpeza:
id_venda                   int64
data_venda        datetime64[ns]
cliente                   object
produto                   object
categoria                 object
regiao                    object
quantidade                 int64
preco_unitario           float64
dtype: object


In [143]:

assert pd.api.types.is_datetime64_any_dtype(
    df_limpo["data_venda"]
)

assert pd.api.types.is_integer_dtype(
    df_limpo["quantidade"]
)

assert pd.api.types.is_float_dtype(
    df_limpo["preco_unitario"]
)

assert df_limpo["data_venda"].notna().all()

print("RF03: validações concluídas com sucesso!")


RF03: validações concluídas com sucesso!


# 06 - Análise exploratória

## Colunas Derivadas

In [144]:

# RF04 - Criar colunas derivadas

# Trabalhar a partir do dataset limpo
df_analise = df_limpo.copy()

# 1. Receita total por item
df_analise["receita_total"] = (
    df_analise["quantidade"] * df_analise["preco_unitario"]
)

# 2. Mês numérico
df_analise["mes"] = df_analise["data_venda"].dt.month

# 3. Nome do mês em português
meses = {
    1: "Janeiro",
    2: "Fevereiro",
    3: "Março",
    4: "Abril",
    5: "Maio",
    6: "Junho",
    7: "Julho",
    8: "Agosto",
    9: "Setembro",
    10: "Outubro",
    11: "Novembro",
    12: "Dezembro"
}

df_analise["mes_nome"] = df_analise["mes"].map(meses)

# 4. Trimestre
df_analise["trimestre"] = (
    "Q" + df_analise["data_venda"].dt.quarter.astype(str)
)

# 5. Ano
df_analise["ano"] = df_analise["data_venda"].dt.year

# 6. Classificação condicional da receita por item
condicoes = [
    df_analise["receita_total"] < 500,
    (
        (df_analise["receita_total"] >= 500)
        & (df_analise["receita_total"] < 5000)
    ),
    df_analise["receita_total"] >= 5000
]

faixas = [
    "Baixo Valor",
    "Medio Valor",
    "Alto Valor"
]

df_analise["faixa_receita_item"] = "Nao Classificado"

for condicao, faixa in zip(condicoes, faixas):
    df_analise.loc[
        condicao, "faixa_receita_item"
    ] = faixa

print("RF04: colunas derivadas criadas com sucesso!")

display(df_analise.head())


RF04: colunas derivadas criadas com sucesso!


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario,receita_total,mes,mes_nome,trimestre,ano,faixa_receita_item
0,1,2025-05-21,Cliente_016,Mouse,Perifericos,Sudeste,2,102.90,205.80,5,Maio,Q2,2025,Baixo Valor
2,3,2025-10-15,Cliente_013,Smartphone,Celulares,Norte,9,2267.33,20405.97,10,Outubro,Q4,2025,Alto Valor
3,4,2025-05-23,Cliente_001,Monitor,Computadores,Sul,4,1181.72,4726.88,5,Maio,Q2,2025,Medio Valor
4,5,2025-03-21,Cliente_014,Mouse,Perifericos,Nordeste,7,114.25,799.75,3,Março,Q1,2025,Medio Valor
5,6,2025-07-03,Cliente_023,Notebook,Computadores,Sudeste,2,3373.92,6747.84,7,Julho,Q3,2025,Alto Valor


In [145]:

# Verificar se todas as colunas exigidas existem
colunas_rf04 = [
    "receita_total",
    "mes",
    "mes_nome",
    "trimestre",
    "ano",
    "faixa_receita_item"
]

assert all(
    coluna in df_analise.columns
    for coluna in colunas_rf04
)

# Conferir os valores derivados
display(
    df_analise[
        [
            "data_venda",
            "quantidade",
            "preco_unitario",
            "receita_total",
            "mes",
            "mes_nome",
            "trimestre",
            "ano",
            "faixa_receita_item"
        ]
    ].head(10)
)

print("Todas as colunas obrigatórias da RF04 foram criadas.")


,data_venda,quantidade,preco_unitario,receita_total,mes,mes_nome,trimestre,ano,faixa_receita_item
0,2025-05-21,2,102.90,205.80,5,Maio,Q2,2025,Baixo Valor
2,2025-10-15,9,2267.33,20405.97,10,Outubro,Q4,2025,Alto Valor
3,2025-05-23,4,1181.72,4726.88,5,Maio,Q2,2025,Medio Valor
4,2025-03-21,7,114.25,799.75,3,Março,Q1,2025,Medio Valor
5,2025-07-03,2,3373.92,6747.84,7,Julho,Q3,2025,Alto Valor
6,2025-07-13,8,121.30,970.40,7,Julho,Q3,2025,Medio Valor
7,2025-12-27,6,358.12,2148.72,12,Dezembro,Q4,2025,Medio Valor
8,2025-04-30,5,400.95,2004.75,4,Abril,Q2,2025,Medio Valor
9,2025-07-01,6,107.86,647.16,7,Julho,Q3,2025,Medio Valor
10,2025-05-06,3,252.56,757.68,5,Maio,Q2,2025,Medio Valor


Todas as colunas obrigatórias da RF04 foram criadas.


## Calcular Métricas Agregadas com groupby

In [146]:
## Calcular Métricas Agregadas com groupby
## deverá calcular e exibir métricas agrupadas em, no mínimo, quatro dimensões

def calcular_metricas(df):
    """
    Calcula as métricas agregadas do dataset de vendas.

    Retorna:
        Dicionário de DataFrames com as chaves:
        por_mes, top_produtos, por_categoria e por_regiao.
    """

    # Validar as colunas necessárias
    colunas_obrigatorias = [
        "pedido_id",
        "data_venda",
        "mes",
        "produto",
        "categoria",
        "regiao",
        "quantidade",
        "receita_total"
    ]

    colunas_ausentes = [
        coluna for coluna in colunas_obrigatorias
        if coluna not in df.columns
    ]

    if colunas_ausentes:
        raise ValueError(
            f"Colunas ausentes no DataFrame: {colunas_ausentes}"
        )

    metricas = {}

    # 1. Métricas por mês
    metricas["por_mes"] = (
        df.groupby("mes")
        .agg(
            receita_total=("receita_total", "sum"),
            quantidade=("quantidade", "sum"),
            n_vendas=("pedido_id", "nunique")
        )
        .reset_index()
        .sort_values("mes")
    )
    #2. Métricas por mês/ano
    metricas["por_mes"] = (
    df.groupby(["ano", "mes"])
    .agg(
        receita_total=("receita_total", "sum"),
        quantidade=("quantidade", "sum"),
        n_vendas=("pedido_id", "nunique")
    )
    .reset_index()
    .sort_values(["ano", "mes"])
)

    # 3. Top 5 produtos por receita total
    metricas["top_produtos"] = (
        df.groupby("produto")
        .agg(
            receita_total=("receita_total", "sum")
        )
        .reset_index()
        .sort_values(
            "receita_total",
            ascending=False
        )
        .head(5)
        .reset_index(drop=True)
    )

    # 4. Receita total por categoria
    metricas["por_categoria"] = (
        df.groupby("categoria")
        .agg(
            receita_total=("receita_total", "sum")
        )
        .reset_index()
        .sort_values(
            "receita_total",
            ascending=False
        )
    )

    # 5. Receita total por região
    receita_regiao = (
        df.groupby("regiao")
        .agg(
            receita_total=("receita_total", "sum")
        )
    )

    # Calcular o valor total de cada pedido por região
    pedidos_regiao = (
        df.groupby(["regiao", "pedido_id"])
        .agg(
            valor_pedido=("receita_total", "sum")
        )
        .reset_index()
    )

    # Ticket médio = receita total / número de pedidos
    ticket_regiao = (
        pedidos_regiao.groupby("regiao")
        .agg(
            ticket_medio=("valor_pedido", "mean")
        )
    )

    metricas["por_regiao"] = (
        receita_regiao
        .join(ticket_regiao)
        .reset_index()
        .sort_values(
            "receita_total",
            ascending=False
        )
    )

    # Exibir os resultados de forma legível
    for nome, resultado in metricas.items():
        print(f"\n=== {nome.upper().replace('_', ' ')} ===")
        display(resultado.round(2))

    return metricas



In [147]:
# Mapeando id_venda para pedido_id para satisfazer as colunas obrigatórias da função
df_analise["pedido_id"] = df_analise["id_venda"]

metricas = calcular_metricas(df_analise)


=== POR MES ===


,ano,mes,receita_total,quantidade,n_vendas
0,2025,1,121707.77,86,15
1,2025,2,186065.22,84,18
2,2025,3,129138.46,87,19
3,2025,4,68247.67,85,14
4,2025,5,103600.61,75,12
5,2025,6,72067.26,73,16
6,2025,7,129427.58,97,19
7,2025,8,156072.31,107,19
8,2025,9,57509.42,77,14
9,2025,10,181878.41,137,25



=== TOP PRODUTOS ===


,produto,receita_total
0,Notebook,493242.63
1,Smartphone,343721.95
2,Tablet,236519.71
3,Monitor,205212.61
4,Headset,48879.79



=== POR CATEGORIA ===


,categoria,receita_total
1,Computadores,698455.24
0,Celulares,580241.66
2,Perifericos,101513.41



=== POR REGIAO ===


,regiao,receita_total,ticket_medio
0,Centro-Oeste,342981.92,7297.49
2,Norte,305196.63,6782.15
1,Nordeste,302857.88,8653.08
3,Sudeste,266608.04,6665.20
4,Sul,162565.84,6020.96


## Segmentar_clientes

In [148]:

def segmentar_clientes(df):
    # Agrupar por cliente e somar a receita total
    clientes = (
        df.groupby("cliente")
        .agg(receita_total=("receita_total", "sum"))
        .reset_index()
    )

    # Classificar os clientes por faixa de gastos
    clientes["segmento"] = clientes["receita_total"].apply(
        lambda valor: "Bronze" if valor < 5000
        else "Prata" if valor <= 15000
        else "Ouro"
    )

    # Exibir os 10 clientes que mais gastaram
    top_10_clientes = clientes.sort_values(
        by="receita_total",
        ascending=False
    ).head(10)

    print("Top 10 clientes por receita:")
    display(top_10_clientes)

    # Exibir a quantidade de clientes por segmento
    quantidade_segmentos = clientes["segmento"].value_counts()

    print("Quantidade de clientes por segmento:")
    display(quantidade_segmentos)

    # Retornar as tabelas em um dicionário
    return {
        "clientes_segmentados": clientes,
        "top_10_clientes": top_10_clientes,
        "quantidade_segmentos": quantidade_segmentos
    }


In [149]:
resultado_segmentacao = segmentar_clientes(df_analise)

Top 10 clientes por receita:


,cliente,receita_total,segmento
47,Cliente_049,86381.62,Ouro
4,Cliente_006,78524.09,Ouro
7,Cliente_009,72517.27,Ouro
14,Cliente_016,66606.02,Ouro
6,Cliente_008,58695.74,Ouro
18,Cliente_020,54182.88,Ouro
42,Cliente_044,54128.61,Ouro
26,Cliente_028,51710.71,Ouro
36,Cliente_038,44741.97,Ouro
33,Cliente_035,43323.63,Ouro


Quantidade de clientes por segmento:


,count
segmento,
Ouro,35
Prata,9
Bronze,5


##  Funções Reutilizáveis

In [150]:
def processar_coluna(df, coluna, funcao_transformacao, nome_saida=None):
    """
    Aplica uma função de transformação aos valores de uma coluna.

    Parâmetros:
        df: DataFrame de entrada.
        coluna: Nome da coluna que será transformada.
        funcao_transformacao: Função aplicada a cada valor.
        nome_saida: Nome da coluna que armazenará o resultado.

    Retorna:
        DataFrame com a nova coluna transformada.
    """
    df = df.copy()

    if coluna not in df.columns:
        raise ValueError(f"A coluna '{coluna}' não existe no DataFrame.")

    nome_saida = nome_saida or f"{coluna}_transformado"

    df[nome_saida] = df[coluna].apply(funcao_transformacao)

    return df


In [151]:

# Contexto 1: converter a receita para milhares de reais
df_teste = processar_coluna(
    df_analise,
    "receita_total",
    lambda x: round(x / 1000, 2),
    nome_saida="receita_em_milhares"
)

# Contexto 2: classificar o volume vendido
df_teste = processar_coluna(
    df_teste,
    "quantidade",
    lambda q: "Alto Volume" if q > 5 else "Baixo Volume",
    nome_saida="perfil_volume"
)

display(
    df_teste[
        [
            "receita_total",
            "receita_em_milhares",
            "quantidade",
            "perfil_volume"
        ]
    ].head(10)
)


,receita_total,receita_em_milhares,quantidade,perfil_volume
0,205.80,0.21,2,Baixo Volume
2,20405.97,20.41,9,Alto Volume
3,4726.88,4.73,4,Baixo Volume
4,799.75,0.80,7,Alto Volume
5,6747.84,6.75,2,Baixo Volume
6,970.40,0.97,8,Alto Volume
7,2148.72,2.15,6,Alto Volume
8,2004.75,2.00,5,Baixo Volume
9,647.16,0.65,6,Alto Volume
10,757.68,0.76,3,Baixo Volume


# 07 — Gráficos

## Criar Visualizações com Matplotli


In [152]:
# Importar bibclioteca
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [153]:
# Criar a função para gerar as quatro figuras


def gerar_visualizacoes(df, metricas):
    # Configuração visual
    sns.set_theme(style="whitegrid")

    pasta_saida = "outputs/graficos"
    os.makedirs(pasta_saida, exist_ok=True)

    # Paleta de cores definida para o projeto
    azul = "#2563EB"
    verde = "#16A34A"
    laranja = "#EA580C"
    roxo = "#9333EA"
    paleta = [azul, verde, laranja, roxo, "#0891B2", "#DB2777"]

    # Recuperar as tabelas de métricas
    por_mes = metricas["por_mes"].copy()
    top_produtos = metricas["top_produtos"].copy()
    por_regiao = metricas["por_regiao"].copy()

    # Validar as colunas necessárias
    colunas_df = ["quantidade", "receita_total", "categoria"]
    colunas_mes = ["ano", "mes", "receita_total"]
    colunas_produtos = ["produto", "receita_total"]
    colunas_regiao = ["regiao", "receita_total"]

    for coluna in colunas_df:
        if coluna not in df.columns:
            raise ValueError(f"Coluna ausente no DataFrame: {coluna}")

    for coluna in colunas_mes:
        if coluna not in por_mes.columns:
            raise ValueError(
                f"Coluna ausente em por_mes: {coluna}. "
                "Verifique o agrupamento por ano e mês na RF05."
            )

    for coluna in colunas_produtos:
        if coluna not in top_produtos.columns:
            raise ValueError(f"Coluna ausente em top_produtos: {coluna}")

    for coluna in colunas_regiao:
        if coluna not in por_regiao.columns:
            raise ValueError(f"Coluna ausente em por_regiao: {coluna}")

    # Preparar os dados mensais
    por_mes = por_mes.sort_values(["ano", "mes"])
    por_mes["periodo"] = (
        por_mes["ano"].astype(str)
        + "-"
        + por_mes["mes"].astype(str).str.zfill(2)
    )

    # --------------------------------------------------
    # FIGURA 1 - Receita por mês (linha)
    # --------------------------------------------------
    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(
        por_mes["periodo"],
        por_mes["receita_total"],
        color=azul,
        marker="o",
        linewidth=2.5,
        label="Receita total"
    )

    ax.set_title("Receita Total por Mês", fontsize=14)
    ax.set_xlabel("Ano e mês")
    ax.set_ylabel("Receita total (R$)")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()

    fig.tight_layout()
    fig.savefig(
        f"{pasta_saida}/receita_por_mes.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.close(fig)

    # --------------------------------------------------
    # FIGURA 2 - Top 5 produtos por receita (barras)
    # --------------------------------------------------
    top_5 = (
        top_produtos
        .sort_values("receita_total", ascending=False)
        .head(5)
        .sort_values("receita_total")
    )

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.barh(
        top_5["produto"].astype(str),
        top_5["receita_total"],
        color=verde,
        label="Receita por produto"
    )

    ax.set_title("Top 5 Produtos por Receita", fontsize=14)
    ax.set_xlabel("Receita total (R$)")
    ax.set_ylabel("Produto")
    ax.legend()

    fig.tight_layout()
    fig.savefig(
        f"{pasta_saida}/top_produtos.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.close(fig)

    # --------------------------------------------------
    # FIGURA 3 - Quantidade versus receita (dispersão)
    # --------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))

    categorias = df["categoria"].dropna().unique()
    cores = sns.color_palette("colorblind", n_colors=len(categorias))

    for i, categoria in enumerate(categorias):
        dados_categoria = df[df["categoria"] == categoria]

        ax.scatter(
            dados_categoria["quantidade"],
            dados_categoria["receita_total"],
            color=cores[i],
            label=str(categoria),
            alpha=0.75,
            s=55
        )

    ax.set_title("Quantidade versus Receita por Categoria", fontsize=14)
    ax.set_xlabel("Quantidade vendida")
    ax.set_ylabel("Receita total (R$)")
    ax.legend(title="Categoria", bbox_to_anchor=(1.02, 1), loc="upper left")

    fig.tight_layout()
    fig.savefig(
        f"{pasta_saida}/quantidade_vs_receita.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.close(fig)

    # --------------------------------------------------
    # FIGURA 4 - Painel resumo (2 x 2)
    # --------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Subplot 1: receita mensal
    axes[0, 0].plot(
        por_mes["periodo"],
        por_mes["receita_total"],
        color=azul,
        marker="o",
        linewidth=2
    )
    axes[0, 0].set_title("Receita por Mês")
    axes[0, 0].set_xlabel("Ano e mês")
    axes[0, 0].set_ylabel("Receita (R$)")
    axes[0, 0].tick_params(axis="x", rotation=45)

    # Subplot 2: top 5 produtos
    axes[0, 1].barh(
        top_5["produto"].astype(str),
        top_5["receita_total"],
        color=verde
    )
    axes[0, 1].set_title("Top 5 Produtos")
    axes[0, 1].set_xlabel("Receita (R$)")
    axes[0, 1].set_ylabel("Produto")

    # Subplot 3: quantidade versus receita
    for i, categoria in enumerate(categorias):
        dados_categoria = df[df["categoria"] == categoria]

        axes[1, 0].scatter(
            dados_categoria["quantidade"],
            dados_categoria["receita_total"],
            color=cores[i],
            label=str(categoria),
            alpha=0.75,
            s=40
        )

    axes[1, 0].set_title("Quantidade versus Receita")
    axes[1, 0].set_xlabel("Quantidade")
    axes[1, 0].set_ylabel("Receita (R$)")
    axes[1, 0].legend(title="Categoria", fontsize=8)

    # Subplot 4: receita por região
    dados_regiao = por_regiao.sort_values("receita_total")

    axes[1, 1].barh(
        dados_regiao["regiao"].astype(str),
        dados_regiao["receita_total"],
        color=laranja
    )
    axes[1, 1].set_title("Receita por Região")
    axes[1, 1].set_xlabel("Receita (R$)")
    axes[1, 1].set_ylabel("Região")

    fig.suptitle("SalesInsight PY - Painel Resumo", fontsize=16)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    fig.savefig(
        f"{pasta_saida}/painel_resumo.png",
        dpi=100,
        bbox_inches="tight"
    )
    plt.close(fig)

    print("As quatro figuras foram geradas e salvas em:", pasta_saida)


In [154]:
gerar_visualizacoes(df_analise, metricas)

As quatro figuras foram geradas e salvas em: outputs/graficos


In [155]:
import os

for arquivo in os.listdir("outputs/graficos"):
    print(arquivo)

painel_resumo.png
quantidade_vs_receita.png
receita_por_mes.png
top_produtos.png


In [156]:

import shutil
from google.colab import files

# Compactar a pasta com os gráficos
shutil.make_archive(
    "graficos_salesinsight",
    "zip",
    "outputs/graficos"
)

# Baixar o arquivo ZIP para o computador
files.download("graficos_salesinsight.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [157]:

def main():
    """Executa o fluxo completo do SalesInsight PY."""

    print("=" * 60)
    print("SALESINSIGHT PY - ANÁLISE DE DADOS DE VENDAS")
    print("=" * 60)

    # Etapa 0: garantir que o dataset exista
    if not os.path.exists("vendas.csv"):
        gerar_dataset_vendas().to_csv(
            "vendas.csv",
            index=False,
            encoding="utf-8-sig"
        )
        print("Dataset gerado: vendas.csv")

    # Criar o analisador
    analisador = AnalisadorDeVendas()

    # 1. Carregamento
    analisador.carregar("vendas.csv")

    # 2. Inspeção inicial
    analisador.inspecionar()

    # 3. Limpeza e transformações
    analisador.transformar()

    # 4. Métricas e segmentação
    analisador.analisar()

    # 5. Geração dos gráficos
    analisador.visualizar()

    # 6. Exportação dos resultados
    analisador.resumo()

    print("\n[CONCLUÍDO] Fluxo finalizado com sucesso.")


if __name__ == "__main__":
    main()


SALESINSIGHT PY - ANÁLISE DE DADOS DE VENDAS
Dataset carregado: (200, 8)

=== INSPEÇÃO INICIAL DO DATASET ===
Dimensões: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade          int64
preco_unitario    float64
dtype: object

Valores ausentes por coluna:
id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        0
preco_unitario    0
dtype: int64

Primeiros registros:
   id_venda     data_venda      cliente     produto     categoria    regiao  \
0         1     2025-05-21  Cliente_016       Mouse   Perifericos   Sudeste   
1         2  DATA INVALIDA  Cliente_003       Mouse   Perifericos       Sul   
2         3     2025-10-15  Cliente_013  Smartphone    

,ano,mes,receita_total,quantidade,n_vendas
0,2025,1,121707.77,86,15
1,2025,2,186065.22,84,18
2,2025,3,129138.46,87,19
3,2025,4,68247.67,85,14
4,2025,5,103600.61,75,12
5,2025,6,72067.26,73,16
6,2025,7,129427.58,97,19
7,2025,8,156072.31,107,19
8,2025,9,57509.42,77,14
9,2025,10,181878.41,137,25



=== TOP PRODUTOS ===


,produto,receita_total
0,Notebook,493242.63
1,Smartphone,343721.95
2,Tablet,236519.71
3,Monitor,205212.61
4,Headset,48879.79



=== POR CATEGORIA ===


...

Top 10 clientes por receita:


,cliente,receita_total,segmento
47,Cliente_049,86381.62,Ouro
4,Cliente_006,78524.09,Ouro
7,Cliente_009,72517.27,Ouro
14,Cliente_016,66606.02,Ouro
6,Cliente_008,58695.74,Ouro
18,Cliente_020,54182.88,Ouro
42,Cliente_044,54128.61,Ouro
26,Cliente_028,51710.71,Ouro
36,Cliente_038,44741.97,Ouro
33,Cliente_035,43323.63,Ouro


Quantidade de clientes por segmento:


,count
segmento,
Ouro,35
Prata,9
Bronze,5


Análise concluída.
As quatro figuras foram geradas e salvas em: outputs/graficos
Arquivos CSV e JSON exportados com sucesso!

Estatísticas gerais recuperadas do JSON:
{
    "total_registros": 194.0,
    "total_clientes": 49.0,
    "total_produtos": 7.0,
    "receita_total": 1380210.31,
    "quantidade_total_vendida": 1035.0,
    "ticket_medio": 7114.49
}

[CONCLUÍDO] Fluxo finalizado com sucesso.


# 08 — Execução e resultados

In [166]:

class AnalisadorDeVendas:
    """Coordena o fluxo completo do SalesInsight PY."""

    def __init__(self):
        self.df_bruto = None
        self.df_limpo = None
        self.df_analise = None
        self.relatorio_limpeza = None
        self.metricas = None
        self.resultado_segmentacao = None
        self.estatisticas = None

    def carregar(self, caminho="vendas.csv"):
        """Carrega o dataset bruto."""
        self.df_bruto = pd.read_csv(caminho)
        print(f"Dataset carregado: {self.df_bruto.shape}")
        return self.df_bruto

    def inspecionar(self):
        """Inspeciona o dataset bruto."""
        return inspecionar_dados(self.df_bruto)

    def transformar(self):
        """Limpa os dados antes de realizar transformações."""
        self.df_limpo, self.relatorio_limpeza = tratar_dados(
            self.df_bruto
        )

        self.df_analise = self.df_limpo.copy()

        # Cria as colunas derivadas
        self.df_analise["receita_total"] = (
            self.df_analise["quantidade"]
            * self.df_analise["preco_unitario"]
        )

        self.df_analise["mes"] = (
            self.df_analise["data_venda"].dt.month
        )

        meses = {
            1: "Janeiro", 2: "Fevereiro", 3: "Março",
            4: "Abril", 5: "Maio", 6: "Junho",
            7: "Julho", 8: "Agosto", 9: "Setembro",
            10: "Outubro", 11: "Novembro", 12: "Dezembro"
        }

        self.df_analise["mes_nome"] = (
            self.df_analise["mes"].map(meses)
        )

        self.df_analise["trimestre"] = (
            "Q" + self.df_analise["data_venda"]
            .dt.quarter.astype(str)
        )

        self.df_analise["ano"] = (
            self.df_analise["data_venda"].dt.year
        )

        self.df_analise["faixa_receita_item"] = np.select(
            [
                self.df_analise["receita_total"] < 500,
                self.df_analise["receita_total"] < 5000
            ],
            [
                "Baixo Valor",
                "Medio Valor"
            ],
            default="Alto Valor"
        )

        # Compatibiliza o identificador com a função de métricas
        self.df_analise["pedido_id"] = (
            self.df_analise["id_venda"]
        )

        print("Limpeza e transformações concluídas.")
        return self.df_analise

    def analisar(self):
        """Calcula métricas, segmentação e estatísticas."""
        self.metricas = calcular_metricas(self.df_analise)

        self.resultado_segmentacao = segmentar_clientes(
            self.df_analise
        )

        self.estatisticas = {
            "total_registros": int(len(self.df_analise)),
            "total_clientes": int(
                self.df_analise["cliente"].nunique()
            ),
            "total_produtos": int(
                self.df_analise["produto"].nunique()
            ),
            "receita_total": float(
                self.df_analise["receita_total"].sum()
            ),
            "quantidade_total_vendida": int(
                self.df_analise["quantidade"].sum()
            ),
            "ticket_medio": float(
                self.df_analise.groupby("pedido_id")[
                    "receita_total"
                ].sum().mean()
            )
        }

        print("Análise concluída.")
        return self.metricas

    def visualizar(self):
        """Gera os gráficos a partir dos dados já tratados."""
        gerar_visualizacoes(
            self.df_analise,
            self.metricas
        )

    def resumo(self):
        """Exporta os resultados finais."""
        clientes = self.resultado_segmentacao[
            "clientes_segmentados"
        ]

        exportar_resultados(
            metricas=self.metricas,
            clientes=clientes,
            estatisticas=self.estatisticas
        )


In [167]:

import json
import os
import pandas as pd


def exportar_resultados(metricas, clientes, estatisticas):
    """Exporta os resultados do projeto em arquivos CSV e JSON."""

    # Cria a pasta de saída, caso ela ainda não exista
    os.makedirs("outputs", exist_ok=True)

    # 1. Exporta as métricas por mês para CSV
    metricas["por_mes"].to_csv(
        "outputs/metricas_por_mes.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # 2. Exporta a segmentação de clientes para CSV
    clientes.to_csv(
        "outputs/segmentacao_clientes.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # 3. Prepara as estatísticas para gravação em JSON
    serializavel = {
        chave: round(float(valor), 2)
        for chave, valor in estatisticas.items()
    }

    caminho = "outputs/estatisticas_gerais.json"

    # Grava as estatísticas em JSON
    with open(caminho, "w", encoding="utf-8") as arquivo:
        json.dump(
            serializavel,
            arquivo,
            indent=4,
            ensure_ascii=False
        )

    print("Arquivos CSV e JSON exportados com sucesso!")

    # 4. Lê o JSON gravado para comprovar a operação
    with open(caminho, "r", encoding="utf-8") as arquivo:
        conferencia = json.load(arquivo)

    print("\nEstatísticas gerais recuperadas do JSON:")
    print(json.dumps(conferencia, indent=4, ensure_ascii=False))

    return conferencia


In [168]:


# Recuperar a tabela de clientes já segmentados
clientes = resultado_segmentacao["clientes_segmentados"]

display(clientes.head())

print("\nQuantidade de clientes por segmento:")
print(clientes["segmento"].value_counts())


,cliente,receita_total,segmento
0,Cliente_001,22350.08,Ouro
1,Cliente_002,14992.37,Prata
2,Cliente_003,34932.57,Ouro
3,Cliente_005,24718.36,Ouro
4,Cliente_006,78524.09,Ouro



Quantidade de clientes por segmento:
segmento
Ouro      35
Prata      9
Bronze     5
Name: count, dtype: int64


In [169]:

# Calcular estatísticas gerais do projeto

estatisticas = {
    "total_registros": len(df_analise),
    "total_clientes": int(df_analise["cliente"].nunique()),
    "total_produtos": int(df_analise["produto"].nunique()),
    "receita_total": float(df_analise["receita_total"].sum()),
    "quantidade_total_vendida": int(df_analise["quantidade"].sum()),
    "ticket_medio": float(
        df_analise.groupby("pedido_id")["receita_total"].sum().mean()
    )
}

print("Estatísticas gerais calculadas:")
print(estatisticas)


Estatísticas gerais calculadas:
{'total_registros': 194, 'total_clientes': 49, 'total_produtos': 7, 'receita_total': 1380210.3099999998, 'quantidade_total_vendida': 1035, 'ticket_medio': 7114.486134020618}


In [170]:

resultado_json = exportar_resultados(
    metricas=metricas,
    clientes=clientes,
    estatisticas=estatisticas
)


Arquivos CSV e JSON exportados com sucesso!

Estatísticas gerais recuperadas do JSON:
{
    "total_registros": 194.0,
    "total_clientes": 49.0,
    "total_produtos": 7.0,
    "receita_total": 1380210.31,
    "quantidade_total_vendida": 1035.0,
    "ticket_medio": 7114.49
}


In [171]:

import os
import pandas as pd
import json

print("=== ARQUIVOS EXPORTADOS ===")

for arquivo in [
    "outputs/metricas_por_mes.csv",
    "outputs/segmentacao_clientes.csv",
    "outputs/estatisticas_gerais.json"
]:
    print(f"{arquivo}: {'OK' if os.path.exists(arquivo) else 'NÃO ENCONTRADO'}")

# Conferir a leitura dos CSVs
metricas_lidas = pd.read_csv("outputs/metricas_por_mes.csv")
clientes_lidos = pd.read_csv("outputs/segmentacao_clientes.csv")

print("\nMétricas mensais exportadas:")
display(metricas_lidas.head())

print("\nSegmentação exportada:")
display(clientes_lidos.head())

# Conferir a leitura do JSON
with open("outputs/estatisticas_gerais.json", "r", encoding="utf-8") as arquivo:
    estatisticas_lidas = json.load(arquivo)

print("\nEstatísticas gerais lidas do JSON:")
print(json.dumps(estatisticas_lidas, indent=4, ensure_ascii=False))


=== ARQUIVOS EXPORTADOS ===
outputs/metricas_por_mes.csv: OK
outputs/segmentacao_clientes.csv: OK
outputs/estatisticas_gerais.json: OK

Métricas mensais exportadas:


,ano,mes,receita_total,quantidade,n_vendas
0,2025,1,121707.77,86,15
1,2025,2,186065.22,84,18
2,2025,3,129138.46,87,19
3,2025,4,68247.67,85,14
4,2025,5,103600.61,75,12



Segmentação exportada:


,cliente,receita_total,segmento
0,Cliente_001,22350.08,Ouro
1,Cliente_002,14992.37,Prata
2,Cliente_003,34932.57,Ouro
3,Cliente_005,24718.36,Ouro
4,Cliente_006,78524.09,Ouro



Estatísticas gerais lidas do JSON:
{
    "total_registros": 194.0,
    "total_clientes": 49.0,
    "total_produtos": 7.0,
    "receita_total": 1380210.31,
    "quantidade_total_vendida": 1035.0,
    "ticket_medio": 7114.49
}


#  09- Executar o Fluxo Completo

In [172]:
import os

def main():
    print("=" * 55)
    print("SALESINSIGHT PY - ANÁLISE DE DADOS DE VENDAS")
    print("=" * 55)

    # 1. Garantir que o dataset exista
    if not os.path.exists("vendas.csv"):
        gerar_dataset_vendas().to_csv(
            "vendas.csv",
            index=False,
            encoding="utf-8-sig"
        )
        print("Dataset vendas.csv criado.")

    # 2. Instanciar a classe
    analisador = AnalisadorDeVendas()

    # 3. Carregar os dados
    analisador.carregar()

    # 4. Inspecionar os dados brutos
    analisador.inspecionar()

    # 5. Limpar e transformar os dados
    analisador.transformar()

    # 6. Calcular métricas e segmentar clientes
    analisador.analisar()

    # 7. Gerar os gráficos
    analisador.visualizar()

    # 8. Exportar os resultados em CSV e JSON
    analisador.resumo()

    print("\n[CONCLUÍDO] Fluxo finalizado com sucesso.")

if __name__ == "__main__":
    main()

SALESINSIGHT PY - ANÁLISE DE DADOS DE VENDAS
Dataset carregado: (200, 8)

=== INSPEÇÃO INICIAL DO DATASET ===
Dimensões: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda         object
cliente            object
produto            object
categoria          object
regiao             object
quantidade          int64
preco_unitario    float64
dtype: object

Valores ausentes por coluna:
id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        0
preco_unitario    0
dtype: int64

Primeiros registros:
   id_venda     data_venda      cliente     produto     categoria    regiao  \
0         1     2025-05-21  Cliente_016       Mouse   Perifericos   Sudeste   
1         2  DATA INVALIDA  Cliente_003       Mouse   Perifericos       Sul   
2         3     2025-10-15  Cliente_013  Smartphone    

,ano,mes,receita_total,quantidade,n_vendas
0,2025,1,121707.77,86,15
1,2025,2,186065.22,84,18
2,2025,3,129138.46,87,19
3,2025,4,68247.67,85,14
4,2025,5,103600.61,75,12
5,2025,6,72067.26,73,16
6,2025,7,129427.58,97,19
7,2025,8,156072.31,107,19
8,2025,9,57509.42,77,14
9,2025,10,181878.41,137,25



=== TOP PRODUTOS ===


,produto,receita_total
0,Notebook,493242.63
1,Smartphone,343721.95
2,Tablet,236519.71
3,Monitor,205212.61
4,Headset,48879.79



=== POR CATEGORIA ===


...

Top 10 clientes por receita:


,cliente,receita_total,segmento
47,Cliente_049,86381.62,Ouro
4,Cliente_006,78524.09,Ouro
7,Cliente_009,72517.27,Ouro
14,Cliente_016,66606.02,Ouro
6,Cliente_008,58695.74,Ouro
18,Cliente_020,54182.88,Ouro
42,Cliente_044,54128.61,Ouro
26,Cliente_028,51710.71,Ouro
36,Cliente_038,44741.97,Ouro
33,Cliente_035,43323.63,Ouro


Quantidade de clientes por segmento:


,count
segmento,
Ouro,35
Prata,9
Bronze,5


Análise concluída.
As quatro figuras foram geradas e salvas em: outputs/graficos
Arquivos CSV e JSON exportados com sucesso!

Estatísticas gerais recuperadas do JSON:
{
    "total_registros": 194.0,
    "total_clientes": 49.0,
    "total_produtos": 7.0,
    "receita_total": 1380210.31,
    "quantidade_total_vendida": 1035.0,
    "ticket_medio": 7114.49
}

[CONCLUÍDO] Fluxo finalizado com sucesso.
